## Model Trainer

In [ ]:
import os
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\TextSummarizer'

## Model Trainer
`src/textsummarizer/entity/config_entity.py`

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int

`src/textsummarizer/config/configuration.py`

In [3]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.main import read_yaml, create_directories

[ 2026-05-13 22:52:54,847 : INFO : __init__ : Logger initialized successfully ]


In [4]:
class ConfigurationManager:
    def __init__(self, 
                 config_path=CONFIG_FILE_PATH,
                 params_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir= config.root_dir,
            data_path= config.data_path,
            model_ckpt= config.model_ckpt,
            num_train_epochs= params.num_train_epochs,
            warmup_steps= params.warmup_steps,
            per_device_train_batch_size= params.per_device_train_batch_size,
            weight_decay= params.weight_decay,
            logging_steps= params.logging_steps,
            evaluation_strategy= params.eval_strategy,
            eval_steps= params.eval_steps,
            save_steps= params.save_steps,
            gradient_accumulation_steps= params.gradient_accumulation_steps,
            per_device_eval_batch_size= params.per_device_eval_batch_size
        )

## Components
`src/textsummarizer/components/modeltrainer.py`

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

c:\Users\singh\OneDrive\Desktop\TextSummarizer\TSenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        ## Loading the data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
                        output_dir=self.config.root_dir,
                        num_train_epochs= self.config.num_train_epochs,
                        warmup_steps= self.config.warmup_steps,
                        per_device_train_batch_size= self.config.per_device_train_batch_size,
                        per_device_eval_batch_size= self.config.per_device_eval_batch_size,
                        weight_decay=self.config.weight_decay,
                        logging_steps=self.config.logging_steps,
                        eval_strategy= self.config.evaluation_strategy,
                        eval_steps=self.config.eval_steps,
                        save_steps=self.config.save_steps,
                        gradient_accumulation_steps=self.config.gradient_accumulation_steps
                        )
        trainer = Trainer(
                        model=model_pegasus,
                        args=trainer_args,
                        processing_class=tokenizer,
                        data_collator=seq2seq_data_collator,
                        train_dataset=dataset_samsum_pt["test"],
                        eval_dataset=dataset_samsum_pt["validation"]
                        )
        trainer.train()

        ## Save Model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        ## Save Tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [7]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Found existing installation: transformers 5.8.1
Uninstalling transformers-5.8.1:
  Successfully uninstalled transformers-5.8.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.8.1-py3-none-any.whl (10.6 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config= model_trainer_config)
model_trainer.train()

[ 2026-05-13 22:56:30,154 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-13 22:56:30,161 : INFO : main : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-13 22:56:30,164 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-13 22:56:30,165 : INFO : main : Create Directory at: artifacts/model_trainer ]
[ 2026-05-13 22:56:30,590 : INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect" ]
[ 2026-05-13 22:56:30,618 : INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK" ]
[ 2026-05-13 22:56:30,892 : INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect" ]
[ 2026-05-13 22:56:30,915 : INFO : _client : HTTP Request: HEAD https://huggi

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 9281.55it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[ 2026-05-13 22:56:45,921 : INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect" ]
[ 2026-05-13 22:56:45,953 : INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK" ]


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\singh\OneDrive\Desktop\TextSummarizer\TSenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


c:\Users\singh\OneDrive\Desktop\TextSummarizer\TSenv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\singh\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyboardInterrupt: 